# DPO post-training on GPU or Ascend NPU

This notebook is a runnable starting point for Direct Preference Optimization (DPO) with [LLaMA-Factory](https://github.com/hiyouga/LLaMA-Factory). It uses the same configuration shape on NVIDIA CUDA and Huawei Ascend NPU, with separate accelerator checks and dependency instructions.

The example uses LoRA so that a short smoke run fits on a modest accelerator. Replace the sample model, dataset, and training values before a production run. Keep the notebook, model cache, dataset, checkpoints, and virtual environment on the Workbench or Notebook PVC.

## Supported execution choices

| Accelerator | Recommended runtime | Notes |
| --- | --- | --- |
| NVIDIA GPU | A CUDA image with LLaMA-Factory 0.9.4, or `alaudadockerhub/llamafactory0.9-cu126-amd64:v0.1.0` | Use `bf16` when the GPU supports it; otherwise use `fp16`. See [Training Runtime Images](./training-runtimes.mdx). |
| Ascend NPU | An Ascend CANN image with `torch_npu`, or `alaudadockerhub/llamafactory0.9-cann8.5-arm64:v0.1.0` | Do not replace the image's PyTorch/CANN stack when installing LLaMA-Factory. Validate the exact driver, CANN, and device-plugin combination first. See [Training Runtime Images](./training-runtimes.mdx). |

The notebook runs the CLI in the current container. If your cluster uses Kubeflow `TrainJob` or VolcanoJob, use the generated YAML values as the starting point for the job container instead.

## Before you begin

1. Create a Workbench or Kubeflow Notebook with a persistent home/workspace PVC.
2. Select a GPU or Ascend NPU image with a working PyTorch stack.
3. Make the base model available locally or allow the container to download it from Hugging Face.
4. Prepare a preference dataset with one `prompt`, one better `chosen` answer, and one worse `rejected` answer per record.

DPO is sensitive to the quality and consistency of preference pairs. Start with a small smoke dataset, confirm that the loss decreases and checkpoints are written, then scale the dataset and sequence length.

In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys

# Keep mutable state under the current notebook directory so it lands on the PVC.
WORKDIR = Path.cwd() / "dpo-workspace"
DATASET_DIR = WORKDIR / "dataset"
OUTPUT_DIR = WORKDIR / "dpo-lora"
MERGED_DIR = WORKDIR / "dpo-merged"
CONFIG_PATH = WORKDIR / "dpo.yaml"
MODEL_NAME_OR_PATH = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_NAME = "dpo_demo"
DATASET_FILE = DATASET_DIR / "dpo_demo.jsonl"
DATASET_INFO = DATASET_DIR / "dataset_info.json"

# Set this to "cuda" or "npu" to override auto-detection.
ACCELERATOR = "auto"
INSTALL_LLAMAFACTORY = False  # Set True only when the selected image needs installation.
RUN_TRAINING = False           # Change to True after reviewing dpo.yaml.
RUN_MERGE = False              # Merge the LoRA adapter after a successful run.

WORKDIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Workspace:", WORKDIR)

## Detect the accelerator and check the runtime

The NPU path depends on `torch_npu` being installed by the image. The check below does not install or replace PyTorch.

In [ ]:
import importlib.util
import torch

def detect_accelerator():
    if ACCELERATOR in {"cuda", "npu"}:
        return ACCELERATOR
    if torch.cuda.is_available():
        return "cuda"
    if importlib.util.find_spec("torch_npu") is not None:
        try:
            import torch_npu  # noqa: F401
            if torch.npu.is_available():
                return "npu"
        except Exception as exc:
            print("torch_npu was found but NPU initialization failed:", exc)
    return "cpu"

ACCELERATOR = detect_accelerator()
print("Detected accelerator:", ACCELERATOR)
if ACCELERATOR == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA was selected but no CUDA device is available.")
    print("CUDA devices:", torch.cuda.device_count())
    print("CUDA device 0:", torch.cuda.get_device_name(0))
elif ACCELERATOR == "npu":
    if importlib.util.find_spec("torch_npu") is None:
        raise RuntimeError("NPU was selected but torch_npu is not installed in this image.")
    import torch_npu  # noqa: F401
    if not hasattr(torch, "npu") or not torch.npu.is_available():
        raise RuntimeError("torch_npu is installed but no Ascend NPU is available.")
    print("NPU devices:", torch.npu.device_count())
else:
    raise RuntimeError("No CUDA or Ascend NPU was detected. Select an accelerator-backed notebook image.")

LLAMAFACTORY_CLI = shutil.which("llamafactory-cli")
print("LLaMA-Factory CLI:", LLAMAFACTORY_CLI or "not found")

## Install LLaMA-Factory without replacing the accelerator stack

The documented LLaMA-Factory images already contain the training dependencies, so leave `INSTALL_LLAMAFACTORY = False` in those images. If the current notebook image only has PyTorch and the accelerator libraries, set it to `True` and run this cell.

For NPU, `--no-deps` is intentional: installing a generic PyTorch dependency from PyPI can break the image's CANN/`torch_npu` pairing. Install any missing pure-Python dependencies using the image's approved package source.

In [ ]:
if INSTALL_LLAMAFACTORY and LLAMAFACTORY_CLI is None:
    package = "llamafactory[metrics]==0.9.4"
    command = [sys.executable, "-m", "pip", "install"]
    if ACCELERATOR == "npu":
        command.append("--no-deps")
    command.append(package)
    subprocess.check_call(command)
    LLAMAFACTORY_CLI = shutil.which("llamafactory-cli")

if LLAMAFACTORY_CLI is None:
    raise RuntimeError("llamafactory-cli is unavailable. Select a documented LLaMA-Factory image or enable installation above.")
print("Using:", LLAMAFACTORY_CLI)

## Prepare a DPO preference dataset

LLaMA-Factory maps the `prompt`, `chosen`, and `rejected` columns through `dataset_info.json`. Replace this smoke dataset with your own JSONL before a real run. The prompt and both answers should use the same language and task framing.

In [ ]:
DPO_EXAMPLES = [
    {
        "prompt": "Explain why a persistent volume is useful for a training workbench.",
        "chosen": "A persistent volume keeps notebooks, datasets, checkpoints, and user-installed environments available when the workbench pod is restarted or recreated.",
        "rejected": "A persistent volume makes the GPU faster and removes the need to track model versions.",
    },
    {
        "prompt": "Give one concise safety rule for handling a production secret.",
        "chosen": "Store the secret in the platform secret manager and grant the workload only the permissions it needs.",
        "rejected": "Put the secret directly in the notebook so every training cell can read it.",
    },
]

DATASET_DIR.mkdir(parents=True, exist_ok=True)
with DATASET_FILE.open("w", encoding="utf-8") as handle:
    for row in DPO_EXAMPLES:
        handle.write(json.dumps(row, ensure_ascii=True) + "\n")

dataset_info = {
    DATASET_NAME: {
        "file_name": DATASET_FILE.name,
        "columns": {
            "prompt": "prompt",
            "chosen": "chosen",
            "rejected": "rejected",
        },
    }
}
DATASET_INFO.write_text(json.dumps(dataset_info, indent=2) + "\n", encoding="utf-8")
print(DATASET_FILE)
print(DATASET_INFO.read_text())

In [ ]:
required_columns = {"prompt", "chosen", "rejected"}
with DATASET_FILE.open(encoding="utf-8") as handle:
    rows = [json.loads(line) for line in handle if line.strip()]
assert rows, "The DPO dataset is empty."
assert all(required_columns <= row.keys() for row in rows)
assert all(row["chosen"].strip() != row["rejected"].strip() for row in rows)
print(f"Validated {len(rows)} preference pairs.")

## Create the LLaMA-Factory DPO configuration

The defaults are deliberately small: one epoch, a short context, one sample per device, and LoRA. Increase `cutoff_len`, `num_train_epochs`, and the effective batch size after the smoke run succeeds.

In [ ]:
def yaml_quote(value):
    return json.dumps(str(value))

if ACCELERATOR == "cuda":
    bf16 = torch.cuda.is_bf16_supported()
else:
    # The documented Ascend CANN stack is configured for BF16 training.
    bf16 = ACCELERATOR == "npu"

config_lines = [
    f"model_name_or_path: {yaml_quote(MODEL_NAME_OR_PATH)}",
    "stage: dpo",
    "do_train: true",
    "finetuning_type: lora",
    "lora_target: all",
    "lora_rank: 8",
    "lora_alpha: 16",
    "lora_dropout: 0.05",
    "",
    f"dataset: {DATASET_NAME}",
    f"dataset_dir: {yaml_quote(DATASET_DIR)}",
    "template: qwen",
    "cutoff_len: 1024",
    "max_samples: 1000",
    "overwrite_cache: true",
    "preprocessing_num_workers: 1",
    "",
    "pref_beta: 0.1",
    "pref_loss: sigmoid",
    "per_device_train_batch_size: 1",
    "gradient_accumulation_steps: 8",
    "learning_rate: 5.0e-6",
    "lr_scheduler_type: cosine",
    "logging_steps: 5",
    "save_steps: 100",
    "save_total_limit: 2",
    "num_train_epochs: 1.0",
    "plot_loss: true",
    "overwrite_output_dir: true",
    "save_only_model: true",
    f"output_dir: {yaml_quote(OUTPUT_DIR)}",
    "report_to: none",
    f"bf16: {str(bf16).lower()}",
    f"fp16: {str((not bf16)).lower()}",
    "ddp_timeout: 180000000",
]
CONFIG_PATH.write_text("\n".join(config_lines) + "\n", encoding="utf-8")
print(CONFIG_PATH.read_text())

## Run DPO

Review the model path, dataset, output directory, precision, and sequence length above. Set `RUN_TRAINING = True` only after checking that the selected model template matches your model.

For a multi-GPU CUDA run, use the image's validated `torchrun`/LLaMA-Factory launch convention. For a single-node Ascend run, make the NPU devices visible to the container and use the image's validated process launcher. This notebook uses the single-process CLI so it is also useful for a first smoke test.

In [ ]:
if RUN_TRAINING:
    env = os.environ.copy()
    env.setdefault("TOKENIZERS_PARALLELISM", "false")
    completed = subprocess.run([LLAMAFACTORY_CLI, "train", str(CONFIG_PATH)], cwd=WORKDIR, env=env, check=False)
    if completed.returncode != 0:
        raise RuntimeError(f"LLaMA-Factory exited with code {completed.returncode}. Inspect the logs above.")
else:
    print("Training is disabled. Set RUN_TRAINING = True after reviewing dpo.yaml.")

## Check the adapter output

A successful LoRA run writes adapter weights and tokenizer/configuration files under `OUTPUT_DIR`. Keep this directory on the PVC and copy it to your model storage system before deleting the workbench.

In [ ]:
if OUTPUT_DIR.exists():
    files = sorted(path.relative_to(OUTPUT_DIR) for path in OUTPUT_DIR.rglob("*") if path.is_file())
    print(f"Found {len(files)} output files under {OUTPUT_DIR}")
    for path in files[:30]:
        print(path)
else:
    print("No output directory yet. Run DPO first.")

## Optionally merge the LoRA adapter

The adapter is usually the most portable training artifact. Merge it into the base model only when the target serving runtime needs a complete model directory. The export step may require more host memory than the training step.

In [ ]:
if RUN_MERGE:
    export_config = WORKDIR / "merge.yaml"
    export_lines = [
        f"model_name_or_path: {yaml_quote(MODEL_NAME_OR_PATH)}",
        f"adapter_name_or_path: {yaml_quote(OUTPUT_DIR)}",
        "template: qwen",
        "finetuning_type: lora",
        f"export_dir: {yaml_quote(MERGED_DIR)}",
        "export_size: 5",
        "export_device: cpu",
        "export_legacy_format: false",
    ]
    export_config.write_text("\n".join(export_lines) + "\n", encoding="utf-8")
    completed = subprocess.run([LLAMAFACTORY_CLI, "export", str(export_config)], check=False)
    if completed.returncode != 0:
        raise RuntimeError(f"Adapter merge failed with code {completed.returncode}.")
    print("Merged model:", MERGED_DIR)
else:
    print("Adapter merge is disabled. Set RUN_MERGE = True after a successful training run.")

## GPU and NPU notes

- **GPU:** If `torch.cuda.is_bf16_supported()` is false, the generated config selects FP16. Adjust the batch size and gradient accumulation to fit the available VRAM.
- **NPU:** Keep the CANN, driver, `torch_npu`, and LLaMA-Factory versions from a tested image together. Set the cluster's Ascend device resource and `runtimeClassName: ascend` when the notebook is used to generate a Kubernetes training job.
- **Distributed runs:** The CLI cell is a single-process smoke test. For multiple GPUs or NPUs, use the runtime's tested `torchrun`, HCCL, scheduler, and device visibility settings, then keep the same DPO YAML.
- **Artifacts:** DPO training needs the policy model and a reference policy during training. Only the resulting full model or LoRA adapter is needed for later generation or serving.
- **Evaluation:** Compare the post-trained model with the base model on held-out preference pairs and task quality metrics before publishing it.